# Lecture 24 - Many Files at Once, and Fitting a Line
### EES 3350/5350 · Python in Earth Science · Monday, November 2nd 2026 

Earlier, you read on messy glacier file and combined two pieces into one table. Real projects rarely stop at one or ttwo files: a monitoring network gives you one file per **station**, per **glacier**, per **year**. And once the data is in one table, the next question is usually "how fast is it changing?"

Today:
1. **a whole folder at once**: `Path.glob`, a function, a loop, and `pd.concat`
2. **fitting a line**: `np.polyfit` gives you a rate, and the leftovers tell you whether a line was the right choice

**Learning objectives**
- use `Path` and `.glob()` with a wildcard pattern to list the files you want
- write one function that reads a file the right way, then use it on every file in a loop
- add a column that records which file each row came from, and combine everything with `pd.concat`
- compare groups fairly by choosing a common period
- fit a straight line with `np.polyfit`, draw it with `np.polyval`, and explain what the slope means
- use **residuals** (data minus fit) to judge whether a straight line is a good model

---
## Path 0 : set up

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

data = "../../course-materials/week10/"

---
## Part 1 : a whole folder at once

NVE measured the mass balance of more than just Rembesdalskåka. The folder `week10/glaciers` holds the files for **six** Norwegian glaciers, exactly as they come from NVE's website: one mass balance file and one front file per glacier, plus a readme.

### `Path` and `.glob()`
`Path` (from Python's buil-in `pathlib` module) turns a text path into an object that knows its a path. the `/` operator joins pieces of a path together:

In [ ]:
folder = Path(data) / "glaciers"

print(folder)
print(folder.exists())

`.glob(pattern)` finds every file in a folder whos name matches a **pattern**. In a pattern a `*` is a **wildcard**: it matches any run of characters, including none at all. `.glob()` hands back the matches in no particular order, so wrap it in `sorted()` to get a predictable, alphametical list:

In [ ]:
for path in sorted(folder.glob("*")):
    print(path.name)

Becasue `"*"` matched everything, we printed all the files in that folder.

**A more specific pattern will pick out only what you want:**
|**pattern**|**matches**|**here, that's...**|
|:--|:--|:--|
|`"*"`|every file|all 13 files|
|`"*.csv"`|every name ending in `.csv`|12 files (not the readme)|
|`"massbalanceData_*.csv"`|names starting with `massbalanceData_` and ending in `.csv`|the 6 mass balance files|
|`"*Storbreen*"`|any name with `Storbreen` in it|Storbreen's 2 files|

In [ ]:
mb_files = sorted(folder.glob("massbalanceData_*.csv"))

for path in mb_files:
    print(path.name)

Each item in `mb_files` is a `Path`, and a `Path` can tell you the pieces of its own name. The text method `.split("_")` is new: it breaks a string into a **list** of pieces wherever it finds `_`, and `[-1]` takes the last piece:

In [ ]:
first = mb_files[0]

print(first.name)                    # the whole file name
print(first.stem)                    # the name without the extenstion
print(first.stem.split("_"))         # break the text into a list at each _
print(first.stem.split("_")[-1])     # the last piece: the glacier's name

<div class="alert alert-success">

**Tip:** a good file name carries information. NVE's names hold the kind of data, the glacier's ID number, and the glacier's name, separated by `_`. That's why splitting the name works. (We changed one thing: NVE's file for **Å**lfotbreen came with the Å replaced by an underscore, `__lfotbreen`, so we renamed it `Alfotbreen`.)

</div>

### Step 1 : one function that reads one file correctly

Earlier, it took four tries to read an NVE file. We don't want to repeat those settings six times (and risk a typo in one of them), so put them in a **function**, once:

In [ ]:
def read_nve_massbalance(path):
    """Read one NVE mass balance file and keep the year and the three balance columns."""
    df = pd.read_csv(path, encoding="latin-1", sep=";", decimal=",")
    return df[["Year", "Bw", "Bs", "Ba"]]


read_nve_massbalance(mb_files[0]).head(3)

### Step 2 : loop over the files, and remember where each row came from

Inside the loop, read each file, add a `glacier` column, and collect the DataFrames in a list. **Without that column, once the tables are stacked, there's no way to tell which glacier a row belongs to.**

In [ ]:
pieces = []

for path in mb_files:
    df = read_nve_massbalance(path)
    df["glacier"] = path.stem.split("_")[-1]
    pieces.append(df)

print(len(pieces), "DataFrames in the list")

### Step 3 : stack them with `pd.concat`

`pd.concat` takes exactly what we built: a **list** of DataFrames.

In [ ]:
glaciers = pd.concat(pieces, ignore_index=True)

print(glaciers.shape)
glaciers.sample(5, random_state=3)

`.sample(5)` shows 5 random rows, a quick way to check that rows from different files made it in (`random_state=3` makes it pick the same rows every time).

**The pattern,** which you'll use over and over:
```
list the files --> for each file: read it the same way + label it --> pd.concat the list
```
Now Lecture 19's `groupby` works across all six glaciers at once. A `groupby` line can get long, so let's build one **a step at a time**, saving each piece in a variable and printing it.

**Question:** what year does each glacier's record start?

**Step 1: split the rows into groups.**

In [ ]:
by_glacier = glaciers.groupby("glacier")

print(by_glacier)
print(by_glacier.ngroups, "groups")

That isn't a table! `groupby` only **sorts the rows into groups** (here, 6 of them, one per glacier). It hasn't calculated anything yet, so all Python can print is a description of the object. `.ngroups` tells you how many groups it made.

**Step 2: pick the column you want to summarize.**

In [ ]:
years_by_glacier = by_glacier["Year"]

print(years_by_glacier)

Still not a table: now it's "the `Year` column, sorted into 6 groups." Still nothing calculated.

**Step 3: summarize each group.** This is the step that actually does the work:

In [ ]:
first_year = years_by_glacier.min()

print(first_year)

Now we get a **Series**: one answer per group, with the glacier names as the index.

|**step**|**code**|**what you have**|
|:--|:--|:--|
|1. split|`by_glacier = glaciers.groupby("glacier")`|rows sorted into 6 groups (nothing calculated)|
|2. pick a column|`years_by_glacier = by_glacier["Year"]`|the `Year` column, in 6 groups (nothing calculated)|
|3. summarize|`first_year = years_by_glacier.min()`|a Series: one minimum per glacier|

You'll often see all three steps squeezed onto one line, like `glaciers.groupby("glacier")["Year"].min()`. It does exactly the same thing. **Read a long line like that left to right, one piece at a time**, using this table.

The records start in different years: Storbreen in 1949, Engabreen not until 1970.

<div class="alert alert-success">

**Tip:** when a long line gives an error or a strange answer, break it apart. Save each piece in its own variable and print it. The first piece that looks wrong is where the problem is. There's no prize for fitting everything on one line!

</div>

In [ ]:
print(glaciers.groupby("glacier")["Year"].min())

The records start in different years: Storbreen in 1949, Engabreen not until 1970.

### Comparing fairly: pick a common period
If we added up each glacier's whole record, StorBreen would get 21 more years of adding than Engabreen. To compaire fairly, use the **same years** for every glacier.
First, keep only 1970 onward:

In [ ]:
common = glaciers[glaciers["Year"] >= 1970].copy()

Now a running total **for each glacier**, with the same three steps as before. Only the summary in step 3 changes:

In [ ]:
by_glacier = common.groupby("glacier")    # 1. split
ba_by_glacier = by_glacier["Ba"]          # 2. pick a column
common["cum_Ba"] = ba_by_glacier.cumsum() # 3. a running total within each group

Why not just `common["Ba"].cumsum()`? Look at where one glacier's rows end and the next ones begin: 

In [ ]:
common["cum_plain"] = common["Ba"].cumsum()

common[["Year", "glacier", "Ba", "cum_plain", "cum_Ba"]].iloc[53:57]

A plain `.cumsum()` doesn't know where one glacier ends, so Ålfotbreen's first year **starts from Engabreen's total**. The `groupby` version (`cum_Ba`) restarts at each new glacier, which is what we want.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for name in sorted(common["glacier"].unique()):
    one = common[common["glacier"] == name]
    ax.plot(one["Year"], one["cum_Ba"], label=name)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("year")
ax.set_ylabel("cumulative balance since 1970 (m w.e.)")
ax.set_title("Six Norwegian glaciers")
ax.legend();

`.unique()` lists each different value in a column once, so the loop draws one line per glacier.

To summarize, there are new pieces in that above figure:

1. `common.groupby("glacier")["Ba"].cumsum()` restarts the running total **for each glacier**. A plain `.cumsum()` would keep adding straight from one glacier into the next.
2. `unique()` lists each different value in a column once, so the loop draws one line per glacier.

<div class="alert alert-warning">

**Question:** Which glaciers lost the most since 1970, and which barely changed? Find each glacier on a map of Norway: are the big losers near the coast or far inland?

</div>

<details>
<summary style="color:green;"><strong>Answer</strong></summary>

```python
by_glacier = common.groupby("glacier")
total_change = by_glacier["Ba"].sum()
total_change = total_change.round(1)
total_change = total_change.sort_values()

print(total_change)
```
Hellstugubreen (−29.5 m w.e.) and Storbreen (−25.7 m w.e.) lost by far the most. Both are in **Jotunheimen**, inland, where the climate is drier and more **continental**. Nigardsbreen (+0.1 m w.e.) is almost exactly where it was in 1970. It, Ålfotbreen, Rembesdalskåka, and Engabreen are **maritime** glaciers, closer to the coast, and their very snowy winters around 1989–1995 (the same bump you saw this morning) made up for much of the melting.

</details>

<div class="alert alert-info">

### Exercise 1: comparing least and most snow

1. Find each glacier's **mean** winter balance (`Bw`) and **mean** summer balance (`Bs`) for 1970–2024, using `common`. Round your answer to 2 decimal places.
2. Sort the glaciers by mean winter balance. Which glacier gets the most snow, and which gets the least? How many times more?

    
</div>

In [ ]:
# your code here
# 1
by_glacier = _____________________  # 1. split
seasons = _____________________     # 2. pick two columns
means = _____________________       # 3. summarize
means = _____________________       # 4. round
print(means)

# 2
means = _____________________       # sort by winter balance
print(means)

ratio = _____________________ / _____________________  # how would we grap the winter balance of a particular glacier?  remember `.loc`? 
print("     ")
print(f"{ratio:.1f} times more snow")

---
## Part 2 : fitting a line with `np.polyfit`

*"How fast is it changing?"* is one of the most common questions in Earth science. If the change is steady, the answer is the **slope** of a straight line thorugh the data. For practice, let's use the most famous record in climate science: the yearly average CO$_2$ at Mauna Loa, Hawai'i (the "Keeling curve").

<span style="font-size:0.85em;color:#6b7a8d;"><em>Data: annual mean CO₂ at Mauna Loa Observatory, 1959–2022. Dr. Xin Lan, NOAA Global Monitoring Laboratory (gml.noaa.gov/ccgg/trends/) and Dr. Ralph Keeling, Scripps Institution of Oceanography (scrippsco2.ucsd.edu/).</em></span>

First, let's look at the data:

In [ ]:
!head -3 {data}co2_maunaloa_annual.csv

Okay so it looks like we have years, followed by a value. And they're separated by a comma. This tells us how we need to import it:
- no header
- name columns "year" and "ppm"

In [ ]:
co2 = pd.read_csv(data + "co2_maunaloa_annual.csv", header=None, names=["year", "ppm"])

Now we can plot it.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(co2["year"], co2["ppm"], marker="o", markersize=3)
ax.set_xlabel("year")
ax.set_ylabel("CO$_2$ (ppm)");

### `np.polyfit`: the best straight line

`np.polyfit(x, y, 1)` finds the straight line that comes closest to all the points at once. (Technically, it makes the sum of the squared vertical distances from the points to the line as small as possible, so it's called a **least-squares** fit.) The `1` is the **degree**: a degree-1 polynomial is a straight line, $y = mx + b$.

It gives back an array of the line's two numbers, **slope first**:

In [ ]:
coeffs = np.polyfit(co2["year"], co2["ppm"], 1)
print(coeffs)

We can resave those so we can easily access them (or you can just index into coeffs, if you wanted):

In [ ]:
slope = coeffs[0]
intercept = coeffs[1]
print(f"CO2 rose by about {slope:.2f} ppm per year, on average, from 159 to 2022")

The **slope** has units of (units of y) / (units of x): here, ppm per year. That's the rate.

The intercept, −2882 ppm, is the line's value in the year 0. It's meaningless here (you can't have negative CO$_2$), because the line only describes the years we gave it. Never read a fitted line far outside the data.

### `np.polyval`: drawing the line
`np.polyval(coeffs, x)` plugs each `x` into the line and gives back the fitted `y` values:

In [ ]:
co2["line"] = np.polyval(coeffs, co2["year"])

fig, ax = plt.subplots(figsize=(8, 3.5))

ax.plot(co2["year"], co2["ppm"], marker="o", markersize=3, label="measured")
ax.plot(co2["year"], co2["line"], label=f"straight line ({slope:.2f} ppm/yr)")
ax.set_xlabel("year")
ax.set_ylabel("CO$_2$ (ppm)")
ax.legend();

### Residuals: is a line the right model?

Maybe! But look at the ends, the **residual** is what's left over after the fit: **measured - fitted**. If a straight line is a good model, residuals should bounce randomly around zero with no pattern.

In [ ]:
co2["residual"] = co2["ppm"] - co2["line"]

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(co2["year"], co2["residual"], "o-", markersize=3)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("year")
ax.set_ylabel("residual (ppm)");

The results make **U**: the line is too low at the start and too high in the middle, and too low again at the end. That pattern means the data **curves upward**: CO$_2$ isn't just rising, the *rise itself* is speeding up. A straight line can't show that, so its single slope (1.63 ppm/yr) hids the msot important part of the story.

<div class="alert alert-danger">

**Common Error:** fitting a line and reporting the slope without looking at the residuals. `np.polyfit` will fit a straight line to *anything*, including data that curves, and it never warns you. Always plot the residuals. A pattern in them (a U, a hump, a trend) means the line is missing something real.

</div>

<div class="alert alert-info">

### Exercise 2 : is the rise speeding up?

1. Fit a separate straight line to **1959–1978** and to **2003–2022** (20 years each). What is the rate (slope) in each period?
2. How many times faster was CO₂ rising in the recent period?
3. Plot the full record with both lines drawn **only over their own years**.

</div>

In [ ]:
# your answer here

# 1
early = co2[___________________]                          # how to we make a mask of just 1959 to 1978?
late = co2[___________________]                           # same, but for 2003 to 2022

early_coeffs = np.polyfit(___________________)            # use polyfit to get slope and intercept
late_coeffs = np.polyfit(___________________)             # use polyfit to get slope and intercept

print(f"1959-1978: {___________________:.2f} ppm/yr")                  # printing the slope of the first period
print(f"2003-2022: {___________________:.2f} ppm/yr")                  # printing the slope of the second period

# 2
print(f"{___________________ / ___________________:.1f} times faster") # printing the ratio to figure out how much faster the second period is


# 3
early_line = ___________________      # use polyval to create the early line
late_line = ___________________       # use polyval to create the late line

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(___________________, ___________________, "o", markersize=3, color="gray", label="measured")
ax.plot(___________________, ___________________), linewidth=2, label="1959-1978 fit")
ax.plot(late["year"], np.polyval(late_coeffs, late["year"]), linewidth=2, label="2003-2022 fit")
ax.set_xlabel("year")
ax.set_ylabel("CO$_2$ (ppm)")
ax.legend();

### You can also fit higher degree polynomials! For example...
Let's fit a 2nd degree polynominal (a parabola, parabola, $y = ax^2 + bx + c$) to the full CO₂ record.
```python
np.polyfit(x, y, 2)
```

In [ ]:
coeffs2 = np.polyfit(co2["year"], co2["ppm"], 2)
print(coeffs2)

Running this will give us back **three** numbers:
```
[a               b                c         ]
```
where the highest power coefficient plots first.

Let's plot the residuals to see if the U is gone:

In [ ]:
co2["residual2"] = co2["ppm"] - np.polyval(coeffs2, co2["year"])

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(co2["year"], co2["residual2"], "o-", markersize=3)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("residual, degree 2 (ppm)");

The residuals now wander around zero without the U.

#### We can now compare the **standard deviation** of the residuals (`.std()`) for the degree-1 and degree-2 fits.

In [ ]:
print(co2["residual"].std().round(2), co2["residual2"].std().round(2))

The residual standard deviation drops from about 4.1 ppm (line) to about 0.7 ppm (parabola). The curve captures the speed-up that the line missed.

#### Moral of the story:
A high-degree polynomial will bend to pass close to every wiggle, including year-to-year noise, so the residuals shrink even though the model isn't learning anything real (**overfitting**). It also does wild things outside the data. Choose a model you can **explain physically**: "the growth rate itself is increasing" justifies a curve; "degree 10 had smaller residuals" doesn't.

---
## A useful table

|**you want**|**write**|
|:--|:--|
|a path object|`folder = Path("some/folder")`|
|join pieces of a path|`folder / "file.csv"`|
|every file matching a pattern, in order|`sorted(folder.glob("pattern_*.csv"))`|
|a path's file name / name without extension|`path.name` / `path.stem`|
|read many files the same way|a function, then a `for` loop over the paths|
|remember which file a row came from|`df["source"] = ...` inside the loop|
|stack a list of DataFrames|`pd.concat(list_of_dfs, ignore_index=True)`|
|a random peek at rows|`df.sample(5)`|
|each different value once|`df["COL"].unique()`|
|a running total within each group|`df.groupby("GROUP")["COL"].cumsum()`|
|fit a straight line|`slope, intercept = np.polyfit(x, y, 1)`|
|the fitted values|`np.polyval(coeffs, x)`|
|residuals|`y - np.polyval(coeffs, x)`|

**Now: Lab 10.** You'll read a set of tide gauge files, decide on methods as a team, and compute rates of sea-level change.